In [0]:
from pyspark.sql import functions as F, Window

dbutils.widgets.text("catalog", "dbr_dev", "Unity Catalog")
dbutils.widgets.text("bronze_schema", "live_transit_monitor", "Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("bronze_schema")

SILVER = f"{CATALOG}.{SCHEMA}.gps_positions_silver"

silver = spark.read.table(SILVER)
fact_df = spark.read.table("fact_vehicle_status")
dim_vehicle = spark.read.table("dim_vehicle")
dim_route = spark.read.table("dim_route")
dim_destination = spark.read.table("dim_destination")
dim_time = spark.read.table("dim_time")


In [0]:
@dp_materialized_view(name="gold_period_summary")
def gold_period_summary():
    return(
        fact_df.agg(
            F.countDistinct("vehicleId").alias("total_vehicles"),        # fleet seen over the period
            F.count("*").alias("total_readings"),
            F.round(F.avg(F.when(F.col("has_trip"), F.col("delay_min"))), 2).alias("avg_delay_min"),
            F.round(F.max(F.when(F.col("has_trip"), F.col("delay_min"))), 2).alias("max_delay_min"),
            F.round(F.avg(F.col("is_delayed").cast("int")) * 100, 1).alias("delayed_reading_pct"),  # % of readings delayed
            F.round(F.avg(F.when(F.col("is_moving"), F.col("speed"))), 2).alias("avg_speed"))
    ) 


In [0]:
@dp_materialized_view(name="gold_route_summary")
def gold_route_summary():
    data = (
        fact_df
        .join(dim_route.select("route_key", "routeId", "routeShortName"), on="route_key", how="left")
        .join(dim_vehicle.select("vehicle_key", "transportationtype"), on="vehicle_key", how="left")
    )

    return (
        data.groupBy("routeId", "routeShortName", "transportationType")
        .agg(
            F.countDistinct("vehicleId").alias("active_vehicles"),
            F.round(
                F.avg(F.when(F.col("has_trip"),F.col("delay_min"))),2
            ).alias("avg_delay_min"),

            F.round(
                F.max(F.when(F.col("has_trip"),F.col("delay_min"))),2
            ).alias("max_delay_min"),

            F.round(
                F.avg(F.when( F.col("is_moving"), F.col("speed"))),2
            ).alias("avg_speed"),

            F.countDistinct(
                F.when( F.col("is_delayed"), F.col("vehicleId"))
            ).alias("delayed_vehicles"),

            F.countDistinct(
                F.when(F.col("is_stopped"),F.col("vehicleId") )
            ).alias("stopped_vehicles")
        )
    )
 

In [0]:
#delay distribution
@dp.materialized_view(name="gold_delay_distribution")
def gold_delay_distribution():
    data = (
        fact_df.join(dim_vehicle.select("vehicle_key", "transportationType"), on="vehicle_key", how="left")
    )

    return (
        data
        .filter(F.col("has_trip"))
        .groupBy("delay_bucket", "transportationType")
        .agg(
            F.count("*").alias("records_count"),
            F.countDistinct("vehicle_key").alias("vehicles_count")
        )
    )


In [0]:
@dp.materialized_view(name="gold_destination_summary")
def gold_destination_summary():
    data = (
        fact_df
        .join(dim_route.select("route_key", "routeShortName"), on="route_key", how="left")
        .join(dim_destination.select("destination_key", "headsign"), on="destination_key", how="left")
        .join(dim_vehicle.select("vehicle_key", "transportationType"), on="vehicle_key", how="left")
    )

    return (
        data.filter(
            F.col("has_trip") & F.col("headsign").isNotNull()
        )
        .groupBy(
            "routeShortName", # which line
            "headsign", # destination / direction
            "transportationType"
        )
        .agg(
            F.countDistinct("vehicleId").alias("active_vehicles"),
            F.round(F.avg("delay_min"),2).alias("avg_delay_min"),
            F.round(F.avg(F.when(F.col("is_moving"),F.col("speed"))),2).alias("avg_speed"),
            F.countDistinct(F.when( F.col("is_delayed"),F.col("vehicleId"))).alias("delayed_vehicles")
        )
    )

In [0]:

# latest reading per vehicle -> one point per vehicle for the MAP
@dp.materialized_view(name="gold_fleet_current")
def gold_fleet_current():
    w = Window.partitionBy("vehicleId").orderBy(F.col("event_time_local").desc())
    current = (
        fact_df
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1).drop("rn")
    )

    data = (
        current
        .join(dim_route.select("route_key", "routeShortName"), on="route_key", how="left")
        .join(dim_destination.select("destination_key", "headsign"), on="destination_key", how="left")
        .join(dim_vehicle.select("vehicle_key", "vehicleId", "vehicleCode" ,"transportationType", "model"), on="vehicle_key", how="left")
    )

    return (
        data.select("vehicleId","vehicleCode","routeShortName","headsign",
            "lat","lon","delay","delay_min","delay_bucket","speed","is_moving",
            "transportationType","model", "event_time_local")
    )

In [0]:
@dp.materialized_view(name="gold_live_kpi")
def gold_live_kpi():
    fc = spark.read.table("gold_fleet_current")
    return (
        fc.agg(
            F.max("event_time_local").alias("snapshot_time"),
            F.countDistinct("vehicleId").alias("active_vehicles"),
            F.countDistinct(F.when(F.col("is_moving")==False, F.col("vehicleId"))).alias("stopped_vehicles"),
            F.countDistinct(F.when(F.col("delay")>120, F.col("vehicleId"))).alias("delayed_vehicles"),
            F.round(F.avg("delay_min"),2).alias("avg_delay_min"),
            F.round(F.max("delay_min"),2).alias("max_delay_min")
        )
    )